# Document AI and RAG Prerequisites

This notebook is independent of any application or framework. It teaches the AI concepts needed before building a document-question-answering system.

The lessons move from language models to real Hugging Face embedding models, vector search, prompting, RAG, memory, and evaluation. Run the code cells in order and complete each checkpoint.

## Prerequisites and setup

You should know basic Python: variables, lists, dictionaries, functions, loops, and importing packages.

Install the optional packages before running the model examples:

```bash
pip install sentence-transformers transformers torch numpy scikit-learn
```

The first Hugging Face model download may take a few minutes and requires internet access. After the model is cached, embedding can run locally.

## Lesson 1: Language models and AI applications

A large language model (LLM) predicts likely next tokens from patterns learned during training. It can generate fluent text, but it does not automatically know private documents.

An AI application combines a model with data, instructions, tools, and a workflow. A document AI workflow usually has two phases: index the documents first, then answer questions using retrieved evidence.

Checkpoint: Why can an LLM produce a confident answer even when the answer is not in its training data?

**Answer:** An LLM generates likely text based on learned language patterns. It is optimized to produce a plausible continuation, not to verify every claim against a trusted source. This can lead to a hallucination: a fluent answer that sounds correct but is unsupported.

In [ ]:
question = "What is retrieval augmented generation?"
model_role = "generate an answer from supplied evidence"
print(f"Question: {question}")
print(f"Model role: {model_role}")

## Lesson 2: Tokens, context windows, and model settings

Models process tokens, which may be whole words, word pieces, punctuation, or characters. Every model has a context window that limits the amount of input and output it can handle.

Temperature changes how varied generation can be. Lower temperature is usually preferred for factual answers; higher temperature can produce more variation.

Checkpoint: What can happen if instructions, chat history, and retrieved documents exceed the context window?

**Answer:** The request may fail, be truncated, or omit important information. Even when it succeeds, the model may pay less attention to useful context. Systems manage this by limiting history, selecting only relevant chunks, summarizing older messages, or using a model with a larger context window.

In [ ]:
text = "Embeddings help a computer compare meaning."
rough_token_count = len(text.split())
print(f"Words: {len(text.split())}")
print(f"Rough token estimate: {rough_token_count}")
print("A real tokenizer may split this into a different number of tokens.")

## Lesson 3: Text preparation and chunking

Before text can be searched, it must be extracted, cleaned, and divided into chunks. Chunk size is a tradeoff: larger chunks preserve context, while smaller chunks improve precision and reduce token usage.

Chunk overlap helps preserve meaning when an important sentence crosses a boundary. Metadata such as source name, page number, and section should travel with each chunk.

Checkpoint: Why is sending an entire long document to the model for every question inefficient?

**Answer:** It consumes more input tokens, increases cost and latency, may exceed the model's context window, and can bury the relevant passage in irrelevant text. Chunking allows the system to retrieve only the sections needed for the current question.

In [ ]:
document = """A document contains information about retrieval.

Retrieval finds relevant passages before generation.

The answer should be based on those passages."""
paragraphs = [part.strip() for part in document.split("\n\n") if part.strip()]
print(paragraphs)

## Lesson 4: Real embeddings with Hugging Face

An embedding is a numeric vector representing the meaning of text. We will use the Hugging Face model `sentence-transformers/all-MiniLM-L6-v2` through the `sentence-transformers` library. It is a practical, lightweight starting model for semantic search.

The same embedding model must encode both document chunks and user questions. Similar meanings should have vectors that are close together.

Checkpoint: Why would comparing document vectors created by one model with question vectors created by another model be unreliable?

**Answer:** Different embedding models can use different dimensions, training objectives, and meanings for vector directions. Their vectors are not guaranteed to share the same semantic space, so distance or similarity scores between them may be meaningless. Encode both documents and queries with the same model and preprocessing rules.

In [1]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = [
    "Cats are common household animals.",
    "Kittens are young cats.",
    "Rain is expected tomorrow.",
]
vectors = embedding_model.encode(texts, normalize_embeddings=True)
print(f"Number of texts: {len(vectors)}")
print(f"Embedding dimensions: {vectors.shape[1]}")
print(vectors[0][:5])

e:\Srikanth\Srikanth\Learning\Agentic_Learining\Project_1\DocuMind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1566.88it/s]


Number of texts: 3
Embedding dimensions: 384
[ 0.07666531 -0.00855304  0.02014799  0.0145971  -0.06408554]


## Lesson 5: Similarity and semantic search

Cosine similarity measures the angle between vectors. With normalized embeddings, the dot product is cosine similarity. A larger score means the texts are more semantically similar.

A vector database stores embeddings alongside original text and metadata, then returns the nearest items for a query vector.

Checkpoint: Why should a search result include the original text and metadata, not only the vector?

**Answer:** The vector is only used to find similar items. The language model needs the original text as context, while metadata provides traceability such as the source, page, section, or document ID. Metadata also enables filtering and source citations.

In [2]:
from sklearn.metrics.pairwise import cosine_similarity

query = embedding_model.encode(
    ["A small cat is a pet."],
    normalize_embeddings=True,
)[0]
scores = cosine_similarity([query], vectors)[0]
for text, score in sorted(zip(texts, scores), key=lambda item: item[1], reverse=True):
    print(f"{score:.3f}  {text}")

0.577  Cats are common household animals.
0.564  Kittens are young cats.
-0.060  Rain is expected tomorrow.


## Lesson 6: Vector databases and retrieval settings

A vector database is an index for embeddings. Important retrieval settings include `k`, the number of results; a similarity threshold; metadata filters; and optional reranking.

Retrieval quality depends on chunking, the embedding model, query wording, and the amount of context returned. Retrieving too little misses evidence; retrieving too much adds noise.

Exercise: Change `k` conceptually and predict how precision and recall may change.

## Lesson 7: Prompt engineering and grounding

A grounded prompt separates instructions, retrieved context, and the user question. It should tell the model to use the supplied context, cite evidence when appropriate, and say that the answer is unknown when evidence is missing.

Prompt instructions are not a security boundary. Retrieved documents may contain malicious or irrelevant instructions, so applications must treat document text as untrusted data.

In [ ]:
def build_grounded_prompt(question, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    return (
        "Answer only from the context. If the answer is missing, say you do not know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

print(build_grounded_prompt("What does retrieval do?", ["Retrieval finds relevant passages."]))

## Lesson 8: Retrieval-Augmented Generation (RAG)

RAG combines search with generation:

1. Prepare and index documents.
2. Embed a user question.
3. Retrieve the most relevant chunks.
4. Put the chunks into a grounded prompt.
5. Ask an LLM to generate an answer.

RAG does not retrain the language model. It supplies external context at inference time.

In [ ]:
retrieved_chunks = [
    "RAG retrieves relevant context.",
    "The language model generates an answer from that context.",
]
question = "How does RAG answer questions?"
print(build_grounded_prompt(question, retrieved_chunks))

## Lesson 9: Conversation memory and query rewriting

A follow-up such as `What did it recommend?` may be ambiguous without previous turns. Conversation memory stores earlier messages, while query rewriting converts a follow-up into a standalone search query.

Memory increases useful context but also increases token usage and can carry earlier mistakes forward. Keep only the history needed for the task.

Checkpoint: What is the difference between conversation memory and the document knowledge base?

**Answer:** Conversation memory contains the messages from the current interaction and helps resolve references in follow-up questions. The document knowledge base contains indexed information from source documents. Memory changes as the conversation continues; the knowledge base usually changes only when documents are added, removed, or re-indexed.

In [ ]:
history = [
    {"role": "user", "content": "What is an embedding?"},
    {"role": "assistant", "content": "It is a vector representation of meaning."},
]
follow_up = "How is it used for search?"
standalone_query = "How is an embedding used for semantic search?"
print(f"Follow-up: {follow_up}")
print(f"Rewritten query: {standalone_query}")

## Lesson 10: Evaluation, safety, and failure modes

Evaluate a RAG system at multiple stages:

- Retrieval: Did the returned chunks contain the answer?
- Groundedness: Are claims supported by retrieved text?
- Relevance: Did the answer address the question?
- Robustness: Does it handle missing evidence and adversarial text?
- Operations: Are latency, cost, and model availability acceptable?

Common failures include poor text extraction, bad chunk boundaries, irrelevant retrieval, stale indexes, prompt injection, hallucinations, and overconfident answers. Create a small set of questions with expected evidence before changing the system.

## Readiness checklist

Before starting a document AI project, you should be able to explain:

- How LLMs, tokens, context windows, and temperature work
- How Hugging Face models create real embeddings
- Why the same embedding model is used for documents and queries
- How cosine similarity and vector databases support semantic search
- Why documents are chunked and what metadata should be retained
- How prompts ground an LLM and why grounding is not a complete security boundary
- The difference between retrieval-augmented generation and model fine-tuning
- How conversation memory and query rewriting support follow-up questions
- How to evaluate retrieval, groundedness, answer quality, safety, cost, and latency

Final exercise: choose a small collection of documents, define five questions and their expected evidence, then design an ingestion, retrieval, prompting, and evaluation plan before writing application code.